# File này chia dữ liệu theo x_label có 3 giá trị: 0, 1, 2 tương ứng với train, val, test.

## Bước này tạo file:

- df_inter.label.parquet
- df_inter.label.inter (bỏ vào model train luôn đổi tên lại trong đúng file data.yaml ví dụ baby.inter)


# Train/Validation/Test data splitting

- Based on generated interactions, perform data splitting


In [193]:
import os
import pandas as pd

In [194]:
PATH = "../data/baby/2014"

In [195]:
os.makedirs(os.path.join(PATH, "step2_train_test_split"), exist_ok=True)

## Load interactions


In [196]:
df = pd.read_parquet(os.path.join(PATH, "step1_rating_to_inter", "df_inter.parquet"))

In [197]:
print(f"shape: {df.shape}")
df[:4]

shape: (160792, 6)


,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X
2,2,0,5.0,1395187200,A2LL1TGG90977E,097293751X
3,3,0,5.0,1376697600,A5G19RYX8599E,097293751X


Đoạn code này thực hiện hai thao tác xử lý dữ liệu có vẻ trái ngược nhau nhưng lại rất phổ biến trong quá trình chuẩn bị dữ liệu cho hệ thống gợi ý.


In [198]:
# df = df.sample(frac=1).reset_index(drop=True)

df.sort_values(by=["userID"], inplace=True)
df[:10]

,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
56087,0,1922,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU
54976,0,1879,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6
46605,0,1587,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0
101675,0,3870,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY
43139,1,1434,5.0,1372377600,A19K65VY14D13R,B000YDIGCC
132789,1,5403,5.0,1367452800,A19K65VY14D13R,B006OK476S
128668,1,5176,5.0,1372464000,A19K65VY14D13R,B005SPCXKC
76953,1,2828,5.0,1372464000,A19K65VY14D13R,B002QBDMDI
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X


In [199]:
# 1. Khai báo tên cột
uid_field, iid_field = "userID", "itemID"

# 2. Nhóm dữ liệu (Groupby)
uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [0, 1922, 1879, 1587, 3870]),
 (1, [1434, 5403, 5176, 2828, 0, 1177]),
 (2, [6578, 6735, 2846, 4011, 0])]

In [200]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

In [201]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())
for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:10]

[0, 0, 0, 1, 2, 0, 0, 0, 0, 1]

In [202]:
df["x_label"] = new_label
df[:20]

,userID,itemID,rating,timestamp,reviewerID,asin,x_label
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,0
56087,0,1922,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
54976,0,1879,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,0
46605,0,1587,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
101675,0,3870,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,2
43139,1,1434,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
132789,1,5403,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
128668,1,5176,5.0,1372464000,A19K65VY14D13R,B005SPCXKC,0
76953,1,2828,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,0
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,1


In [203]:
df.to_parquet(
    os.path.join(PATH, "step2_train_test_split", "df_inter.label.parquet"), index=False
)

## Reload


In [204]:
indexed_df = pd.read_parquet(
    os.path.join(PATH, "step2_train_test_split", "df_inter.label.parquet")
)
print(f"shape: {indexed_df.shape}")
indexed_df[:20]

shape: (160792, 7)


,userID,itemID,rating,timestamp,reviewerID,asin,x_label
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,0
1,0,1922,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
2,0,1879,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,0
3,0,1587,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
4,0,3870,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,2
5,1,1434,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
6,1,5403,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
7,1,5176,5.0,1372464000,A19K65VY14D13R,B005SPCXKC,0
8,1,2828,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,0
9,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,1


In [205]:
train_df = indexed_df[["userID", "itemID", "rating", "timestamp", "x_label"]].copy()
train_df

,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1373932800,0
1,0,1922,5.0,1363996800,0
2,0,1879,5.0,1373932800,0
3,0,1587,5.0,1373932800,1
4,0,3870,4.0,1357689600,2
...,...,...,...,...,...
160787,19444,7022,5.0,1393804800,0
160788,19444,6994,5.0,1393632000,0
160789,19444,7023,4.0,1394668800,0
160790,19444,7005,5.0,1396310400,1


In [206]:
# Lưu lại file .inter để train, tùy dataset mà tên bỏ vào thư mục data sẽ khác
# ví dụ data/baby/baby.inter, data/beauty/beauty.inter, data/clothing/clothing.inter
# ghi file trong windows size khác so với linux vì CRLF
train_df.to_csv(
    os.path.join(PATH, "step2_train_test_split", "df_inter.label.inter"),
    sep="\t",
    index=False,
)

In [207]:
train_df.shape

(160792, 5)

In [208]:
train_df.head(3)

,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1373932800,0
1,0,1922,5.0,1363996800,0
2,0,1879,5.0,1373932800,0


In [209]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 160792 entries, 0 to 160791
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userID     160792 non-null  int64  
 1   itemID     160792 non-null  int64  
 2   rating     160792 non-null  float64
 3   timestamp  160792 non-null  int64  
 4   x_label    160792 non-null  int64  
dtypes: float64(1), int64(4)
memory usage: 6.1 MB


In [210]:
u_id_str, i_id_str = "userID", "itemID"
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 19445
# of unique courses: 7050
min/max of unique learners: 0/19444
min/max of unique courses: 0/7049
